# NB06：粗脂肪分析（Soxhlet 溶劑萃取, Crude Fat）

**食品分析實驗 週次 6｜Nielsen's Food Analysis, 6th ed. 對應章節：脂肪 (Fat, Ch.17)**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用秤重公式計算**粗脂肪%**，並區分「以乾樣品計」與「以濕樣品計(濕基)」的差別。
2. 計算重複測量的**相對標準差(RSD%)**，判斷方法的精密度(precision)。
3. 用已知脂肪對照品計算**回收率(recovery)**，判斷方法的準確度(accuracy)。
4. 用溶劑空白(solvent blank)校正萃取瓶殘留，理解準確度與精密度是兩回事。
5. 用全班盲樣(全脂奶粉)看到一個真實的**系統誤差(bias)**：直接 Soxhlet(未水解) 相對 Röse–Gottlieb 參考法與標示值偏低，並解釋原因(脂肪被蛋白質/乳糖包覆)。
6. 用 **Pearson square** 算出牛奶標準化(調整脂肪含量)所需的鮮奶油與脫脂乳比例，並驗算質量平衡。

## 如何使用本筆記本

- 直接「全部執行」（Run all / 執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬教學資料**。
- `load_data()` 支援三種資料來源：
  1. **本機/雲端硬碟路徑**：`load_data("data/nb06_sample.csv")` 或你自己上傳的檔名。
  2. **已發布的 Google 試算表 CSV 連結**：檔案 → 共用 → 發布到網路 → 選 CSV，把網址傳入 `load_data(那個網址)`。
  3. **在 Colab 上傳檔案**：執行 Setup 區塊中被註解掉的 `google.colab.files.upload()` 程式碼。
  4. 若以上都沒有提供，`load_data()` 會自動退回使用本筆記本內嵌的模擬資料字串，確保筆記本在任何環境都能獨立執行。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值。做作業時請換成你們班的實際數據。

## 資料格式 (Data Schema)

延續共用的長格式（long format），本週不需要額外欄位——`item` 直接編碼每一筆秤重/結果是什麼（例如 `chips_flask_empty_g`、`chips_flask_fat_g`），`value` 搭配 `unit` 就能表示質量(g)或百分比(%)：

| 欄位 | 說明 |
|---|---|
| `group` | 組別（1–5）；`0` 代表全班共用或 TA 提供，非特定分組 |
| `student` | 代號（例：`G1`；`TA`／`reference_lab`／`label`／`shared` 代表非特定分組的來源）|
| `item` | 量測項目：見下方各節說明 |
| `rep` | 重複次數（大多數項目本週只做 1 次；`fat_replicate_demo` 例外，rep=1–3）|
| `value` | 量測值 |
| `unit` | `value` 的單位（`g` 或 `%`）|
| `temp_C` | 本週不適用，保留欄位以與其他週次的長格式一致，留空 |
| `note` | 備註欄，說明該筆資料的角色/來源 |

## 1. 環境設定 (Setup)

匯入需要的套件，並定義 `load_data()`。**注意**：所有圖表的座標軸標籤與標題一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼（tofu）。文字說明則使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# This is the exact content of data/nb06_sample.csv, embedded so the
# notebook is self-contained (works even without the data/ folder present).
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note
1,G1,chips_flask_empty_g,1,132.4056,g,,萃取瓶(含止爆石)已烘至恆重
1,G1,chips_sample_dry_g,1,2.9688,g,,已乾燥、磨細的洋芋片樣品(沿用 W2 乾燥樣品)
1,G1,chips_flask_fat_g,1,133.467,g,,萃取＋烘乾至恆重後，瓶+脂肪
2,G2,chips_flask_empty_g,1,132.4069,g,,萃取瓶(含止爆石)已烘至恆重
2,G2,chips_sample_dry_g,1,2.9415,g,,已乾燥、磨細的洋芋片樣品(沿用 W2 乾燥樣品)
2,G2,chips_flask_fat_g,1,133.3981,g,,萃取＋烘乾至恆重後，瓶+脂肪
3,G3,chips_flask_empty_g,1,132.4053,g,,萃取瓶(含止爆石)已烘至恆重
3,G3,chips_sample_dry_g,1,2.9905,g,,已乾燥、磨細的洋芋片樣品(沿用 W2 乾燥樣品)
3,G3,chips_flask_fat_g,1,133.4515,g,,萃取＋烘乾至恆重後，瓶+脂肪
4,G4,chips_flask_empty_g,1,132.4033,g,,萃取瓶(含止爆石)已烘至恆重
4,G4,chips_sample_dry_g,1,3.0264,g,,已乾燥、磨細的洋芋片樣品(沿用 W2 乾燥樣品)
4,G4,chips_flask_fat_g,1,133.4861,g,,萃取＋烘乾至恆重後，瓶+脂肪
5,G5,chips_flask_empty_g,1,132.4051,g,,萃取瓶(含止爆石)已烘至恆重
5,G5,chips_sample_dry_g,1,3.0338,g,,已乾燥、磨細的洋芋片樣品(沿用 W2 乾燥樣品)
5,G5,chips_flask_fat_g,1,133.4811,g,,萃取＋烘乾至恆重後，瓶+脂肪
0,TA_demo,fat_replicate_demo,1,34.8,%,,示範同一樣品三重複，用於教你算 RSD，非全班分組數據
0,TA_demo,fat_replicate_demo,2,35.1,%,,示範同一樣品三重複，用於教你算 RSD，非全班分組數據
0,TA_demo,fat_replicate_demo,3,34.6,%,,示範同一樣品三重複，用於教你算 RSD，非全班分組數據
0,TA,known_fat_reference_measured,1,19.47,%,,已知脂肪對照品，證書標示含量 20.0%
0,TA,solvent_blank_residue_g,1,0.0007,g,,只蒸乾溶劑、不含樣品，理論上應趨近於 0
1,G1,milk_blind_fat_pct,1,20.82,%,,全脂奶粉盲樣，直接 Soxhlet(未水解)
2,G2,milk_blind_fat_pct,1,21.93,%,,全脂奶粉盲樣，直接 Soxhlet(未水解)
3,G3,milk_blind_fat_pct,1,21.37,%,,全脂奶粉盲樣，直接 Soxhlet(未水解)
4,G4,milk_blind_fat_pct,1,21.29,%,,全脂奶粉盲樣，直接 Soxhlet(未水解)
5,G5,milk_blind_fat_pct,1,20.99,%,,全脂奶粉盲樣，直接 Soxhlet(未水解)
0,reference_lab,milk_blind_RG_reference,1,25.7,%,,Röse-Gottlieb 參考法數值，TA/參考實驗室提供，非本班本週測值
0,label,milk_blind_label,1,26.0,%,,產品標示值
0,shared,chips_moisture_pct_wet,1,2.0,%,,取自 W2 該批洋芋片的濕基水分測定
"""

In [ ]:
def load_data(source=None):
    """Load NB06 crude-fat data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note

    Parameters
    ----------
    source : str or None
        - None: try a local "data/nb06_sample.csv" relative path first,
          otherwise fall back to the embedded SAMPLE_CSV string.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link): read
          directly with pandas.read_csv.

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            df = pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")
            df = None
    else:
        local_path = os.path.join("data", "nb06_sample.csv")
        if os.path.exists(local_path):
            df = pd.read_csv(local_path)
        else:
            df = pd.read_csv(io.StringIO(SAMPLE_CSV))

    if df is None:
        df = pd.read_csv(io.StringIO(SAMPLE_CSV))

    df["note"] = df["note"].fillna("")
    return df


df = load_data()
print(f"讀入 {len(df)} 筆資料，涵蓋項目：{sorted(df['item'].unique())}")
df

### （選用）在 Colab 上傳班級實際資料

若在 Google Colab 中執行、且要換成全班實際量測值，取消下方程式碼的註解後執行即可；在非 Colab 環境（例如本機 Jupyter）執行這個 cell 不會有任何作用。

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 粗脂肪 % 計算：洋芋片（各組自己的 Soxhlet 結果）

$$\text{粗脂肪}\% = \frac{W(\text{瓶+脂肪}) - W(\text{空瓶})}{W(\text{樣品})} \times 100$$

樣品是**先乾燥過**才秤入套管的，所以這個 % 是以「乾樣品」為分母（dry-sample basis）。

In [ ]:
chips_wide = df[df["item"].str.startswith("chips_") & (df["item"] != "chips_moisture_pct_wet")].copy()
chips_wide = chips_wide.pivot_table(index=["group", "student"], columns="item", values="value").reset_index()
chips_wide["fat_mass_g"] = chips_wide["chips_flask_fat_g"] - chips_wide["chips_flask_empty_g"]
chips_wide["crude_fat_pct_dry"] = chips_wide["fat_mass_g"] / chips_wide["chips_sample_dry_g"] * 100
chips_wide

## 3. 換算回濕基 (Dry Basis -> Wet Basis)

樣品秤入套管前**已經先乾燥過**，所以第 2 節算出的 % 是「以乾樣品為分母」。要跟原始濕重比較，需要用同一批樣品的水分含量（本週取自 W2 的測定值）做轉換：

$$\text{粗脂肪}\%(\text{濕基}) = \text{粗脂肪}\%(\text{乾基}) \times (1 - \text{水分}\%)$$

In [ ]:
moisture_pct_wet = df.loc[df["item"] == "chips_moisture_pct_wet", "value"].iloc[0]
dry_matter_frac = 1 - moisture_pct_wet / 100

chips_wide["crude_fat_pct_wet"] = chips_wide["crude_fat_pct_dry"] * dry_matter_frac
print(f"該批洋芋片水分(wet basis, 取自 W2) = {moisture_pct_wet:.2f}%")
chips_wide[["group", "student", "crude_fat_pct_dry", "crude_fat_pct_wet"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
width = 0.35
x = np.arange(len(chips_wide))
ax.bar(x - width / 2, chips_wide["crude_fat_pct_dry"], width, label="dry basis", color="tab:blue")
ax.bar(x + width / 2, chips_wide["crude_fat_pct_wet"], width, label="wet basis", color="tab:orange")
ax.set_xticks(x)
ax.set_xticklabels(chips_wide["student"])
ax.set_ylabel("Crude fat %")
ax.set_xlabel("Group")
ax.set_title("Potato chips: crude fat, dry vs wet basis")
ax.legend()
plt.tight_layout()
plt.show()

## 4. 精密度：重複性 RSD

同一樣品重複萃取 3 次（教學示範，非全班分組數據——完整的 Soxhlet 萃取耗時，本週每組只做 1 次），計算平均值、標準差(SD)、相對標準差(RSD%)：

$$RSD\% = \frac{SD}{\text{mean}} \times 100\%$$

In [ ]:
rep_demo = df[df["item"] == "fat_replicate_demo"]["value"]
rep_mean = rep_demo.mean()
rep_sd = rep_demo.std(ddof=1)
rep_rsd = rep_sd / rep_mean * 100

print(f"三重複數值：{list(rep_demo.values)}")
print(f"平均 = {rep_mean:.2f}%, SD = {rep_sd:.3f}%, RSD = {rep_rsd:.2f}%")
print("RSD 越小代表方法的精密度越好，但精密度好不代表準確——請對照第 6 節的盲樣偏差。")

## 5. 準確度：已知脂肪對照品回收率、溶劑空白校正

- **準確度(accuracy)**：TA 的已知脂肪對照品（證書標示含量），Soxhlet 測出的值跟證書值比較，算**回收率(recovery)**。
- **溶劑空白(solvent blank)**：只蒸乾溶劑、不含樣品，量殘留重——理論上應趨近於 0；若空白偏高，代表溶劑或玻璃器皿本身有污染，會讓所有樣品的脂肪%虛高，須先扣除。

In [ ]:
ref_known = 20.0  # %, certified value (see note column for provenance)
ref_measured = df.loc[df["item"] == "known_fat_reference_measured", "value"].iloc[0]
ref_recovery = ref_measured / ref_known * 100

solvent_blank_g = df.loc[df["item"] == "solvent_blank_residue_g", "value"].iloc[0]

print(f"已知對照品：證書值 {ref_known:.1f}%，Soxhlet 測得 {ref_measured:.2f}%，回收率 = {ref_recovery:.1f}%")
print(f"溶劑空白殘留 = {solvent_blank_g:.4f} g（理論上應趨近於 0）")
if 95 <= ref_recovery <= 105:
    print("回收率落在 95-105% 之間，整體操作(溶劑、時間、溫度)判定為正常。")
else:
    print("回收率超出 95-105% 範圍，應檢討操作條件。")

## 6. 盲樣的真相：全脂奶粉 Soxhlet 為什麼偏低？

5 組各對同一份全脂奶粉盲樣做**直接 Soxhlet(未水解)**，彙整成班級數據（n=5），跟 Röse–Gottlieb 參考法、標示值比較：奶粉的脂肪被蛋白質與乳糖包覆(encapsulated fat)，溶劑接觸不到，這是一個真實的**系統誤差(bias)**，不是操作失誤。

In [ ]:
milk_blind = df[df["item"] == "milk_blind_fat_pct"]["value"]
milk_blind_mean = milk_blind.mean()
milk_blind_sd = milk_blind.std(ddof=1)

rg_reference = df.loc[df["item"] == "milk_blind_RG_reference", "value"].iloc[0]
label_value = df.loc[df["item"] == "milk_blind_label", "value"].iloc[0]

bias_vs_rg = milk_blind_mean - rg_reference
recovery_vs_rg = milk_blind_mean / rg_reference * 100

print(f"班級直接 Soxhlet 平均 (n={len(milk_blind)}) = {milk_blind_mean:.2f} ± {milk_blind_sd:.2f}% (SD)")
print(f"Röse-Gottlieb 參考法 = {rg_reference:.1f}%　|　標示值 = {label_value:.1f}%")
print(f"bias (Soxhlet - RG) = {bias_vs_rg:+.2f} 個百分點，回收率 = {recovery_vs_rg:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Label", "Rose-Gottlieb\n(reference)", "Soxhlet\n(class mean, n=5)"]
values = [label_value, rg_reference, milk_blind_mean]
errors = [0, 0, milk_blind_sd]
ax.bar(labels, values, yerr=errors, capsize=6, color=["tab:gray", "tab:green", "tab:blue"])
ax.set_ylabel("Fat % in whole milk powder")
ax.set_title("Why direct Soxhlet (no hydrolysis) reads low on milk powder")
plt.tight_layout()
plt.show()

## 7. Pearson Square：牛奶脂肪標準化

工廠把牛奶先脫脂，再依比例加回鮮奶油，調成特定脂肪%的產品。這跟脂肪分析用的是同一套**質量平衡**邏輯：

$$W_{\text{高脂}} = \frac{\text{目標}\% - \text{低脂}\%}{\text{高脂}\% - \text{低脂}\%} \times W_{\text{總}}$$

$$W_{\text{低脂}} = W_{\text{總}} - W_{\text{高脂}}$$

In [ ]:
def pearson_square(high_fat_pct, low_fat_pct, target_fat_pct, total_mass):
    """Two-component Pearson square: mix a high-fat and a low-fat component
    to hit a target fat percentage. Returns (high_mass, low_mass).

    Raises ValueError if target is not strictly between low and high.
    """
    if not (low_fat_pct < target_fat_pct < high_fat_pct):
        raise ValueError("target_fat_pct must be strictly between low_fat_pct and high_fat_pct")
    high_mass = (target_fat_pct - low_fat_pct) / (high_fat_pct - low_fat_pct) * total_mass
    low_mass = total_mass - high_mass
    return high_mass, low_mass


high_fat, low_fat, target_fat, total_kg = 36.0, 0.1, 3.25, 100.0
cream_kg, skim_kg = pearson_square(high_fat, low_fat, target_fat, total_kg)

check_fat_kg = cream_kg * high_fat / 100 + skim_kg * low_fat / 100

print(f"用 {high_fat}% 鮮奶油 + {low_fat}% 脫脂乳，混合 {total_kg} kg 的 {target_fat}% 全脂乳：")
print(f"  鮮奶油 = {cream_kg:.2f} kg")
print(f"  脫脂乳 = {skim_kg:.2f} kg")
print(f"質量平衡驗算：{cream_kg:.2f}x{high_fat}% + {skim_kg:.2f}x{low_fat}% = {check_fat_kg:.3f} kg 脂肪"
      f"（應接近 {target_fat/100*total_kg:.3f} kg）")

## 8. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算）。

### 問題 1

如果忘記做溶劑空白校正（沒有從脂肪重量中扣除第 5 節的溶劑空白殘留重量），你們這組洋芋片的粗脂肪% 會被高估多少（百分點與相對%）？請任選一組資料實際計算。

In [ ]:
# TODO: 請在這裡作答

### 問題 2

用第 6 節班級直接 Soxhlet 平均值，分別計算相對 Röse–Gottlieb 與標示值的 bias 與回收率，並用一到兩句話解釋為什麼方向是「偏低」而不是「偏高」。

In [ ]:
# TODO: 請在這裡作答

### 問題 3

用 `pearson_square()` 函式，改算「用 40% 鮮奶油 + 0.5% 脫脂乳，調配 200 kg 的 2% 低脂乳」各需要多少 kg？並驗算質量平衡。

In [ ]:
# TODO: 請在這裡作答

### 問題 4

如果某組把洋芋片樣品磨得不夠細、顆粒偏大，對粗脂肪%（相對真實值）會有什麼影響（偏高/偏低）？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

已知對照品的回收率（第 5 節算出來的數字）如果落在 95–105% 之外，實驗室 SOP 會認定這批操作不合格。請查看你算出的回收率是否合格；若不合格，列出兩個可能的原因。

In [ ]:
# TODO: 請在這裡作答

## 9. 匯出結果 (Export)

把本週計算結果（洋芋片各組粗脂肪%、盲樣班級平均、QC 回收率）存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔，不會嘗試下載。

In [ ]:
export_chips = chips_wide[["group", "student", "crude_fat_pct_dry", "crude_fat_pct_wet"]].copy()
export_chips["item"] = "potato_chips"

summary_rows = pd.DataFrame([
    {"group": "class_mean_n5", "student": "", "crude_fat_pct_dry": np.nan,
     "crude_fat_pct_wet": milk_blind_mean, "item": "milk_blind_fat_pct_summary"},
    {"group": "qc", "student": "TA", "crude_fat_pct_dry": np.nan,
     "crude_fat_pct_wet": ref_recovery, "item": "known_fat_reference_recovery_pct"},
])

results_df = pd.concat([export_chips, summary_rows], ignore_index=True)

output_path = "nb06_student_results.csv"
results_df.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")